# INFO 4670 / 4760 — Assignment 2
### Cleaning & Integrating the Northgate Data

Raw files are only *read*; every fix is applied to in-memory copies, and the clean table is written to a **new** file (`northgate_clean.csv`). Run top to bottom.

## Setup — load the three files

In [1]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    try:
        from google.colab import files
        print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
        files.upload()
        students = pd.read_csv("student_records.csv")
    except ImportError:   # running locally: fall back to Downloads
        os.chdir(os.path.expanduser("~/Downloads"))
        students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")

# Golden rule: keep pristine copies of the raw data; all cleaning happens on `students` / `enroll` / `activity` in memory.
students_raw = students.copy()
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values

In [2]:
# Which columns have blanks at all?
print(students.isna().sum()[lambda s: s > 0])
n_missing_study = int(students["study_hours_reported"].isna().sum())
print("n_missing_study =", n_missing_study, "| present:", students["study_hours_reported"].notna().sum())

# Is the missingness related to other columns? (decides which of Han's methods is defensible)
miss = students["study_hours_reported"].isna()
print(students.groupby(miss)[["final_gpa", "advisor_meetings", "dropped_out"]].mean().round(3))
print("skew:", round(students["study_hours_reported"].skew(), 3),
      "| corr with final_gpa:", round(students["study_hours_reported"].corr(students["final_gpa"]), 3))

study_hours_reported    255
dtype: int64
n_missing_study = 255 | present: 1772
                      final_gpa  advisor_meetings  dropped_out
study_hours_reported                                          
False                     2.381             2.258        0.199
True                      1.799             1.741        0.290
skew: -0.066 | corr with final_gpa: 0.696


**Your justification:** I use Han's *most probable value* method (regression imputation, predictors `final_gpa` and `advisor_meetings`). The column is symmetric (skew ≈ 0), so a mean would be safe on its own, but the blanks are **not random** — students with missing study hours have a clearly lower mean GPA — and study hours correlate ~0.70 with GPA, so a global mean would push the missing students toward the average and weaken that relationship. Dropping the 255 rows (12.6%) would also bias the sample toward higher-GPA students.

In [3]:
print("BEFORE  n =", len(students), "| non-null:", students["study_hours_reported"].notna().sum(),
      "| mean:", round(students["study_hours_reported"].mean(), 3), "| sd:", round(students["study_hours_reported"].std(), 3))

X_cols = ["final_gpa", "advisor_meetings"]
known = students.loc[~miss]
A = np.column_stack([np.ones(len(known)), known[X_cols].to_numpy()])
beta, *_ = np.linalg.lstsq(A, known["study_hours_reported"].to_numpy(), rcond=None)
pred = np.column_stack([np.ones(miss.sum()), students.loc[miss, X_cols].to_numpy()]) @ beta

students["study_hours_imputed"] = miss.astype(int)          # flag so imputed cells stay traceable
students.loc[miss, "study_hours_reported"] = np.clip(pred.round(1), 0, None)

print("AFTER   n =", len(students), "| non-null:", students["study_hours_reported"].notna().sum(),
      "| mean:", round(students["study_hours_reported"].mean(), 3), "| sd:", round(students["study_hours_reported"].std(), 3))

BEFORE  n = 2027 | non-null: 1772 | mean: 10.486 | sd: 3.865
AFTER   n = 2027 | non-null: 2027 | mean: 10.19 | sd: 3.828


In [4]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories

In [5]:
print("housing BEFORE:"); print(students["housing"].value_counts())
students["housing_clean"] = (students["housing"].str.strip().str.lower()
                             .str.replace(r"[\s\-_]+", " ", regex=True)     # "off-campus" / "off campus" -> same key
                             .map({"on campus": "On-Campus", "off campus": "Off-Campus", "with family": "With Family"}))
assert students["housing_clean"].notna().all()               # nothing fell through the map
print("\nhousing_clean AFTER:"); print(students["housing_clean"].value_counts())

# `major` has the same defect (e.g. "Cyber Security", "DataScience", "Info Technology") - fix it too
print("\nmajor BEFORE:"); print(students["major"].value_counts())
students["major"] = (students["major"].str.lower().str.replace(r"\s+", "", regex=True)
                     .map({"cybersecurity": "Cybersecurity", "datascience": "Data Science",
                           "informationscience": "Information Science", "informationtechnology": "Information Technology",
                           "infotechnology": "Information Technology", "businessanalytics": "Business Analytics",
                           "learningtechnologies": "Learning Technologies"}))
assert students["major"].notna().all()
print("\nmajor AFTER:"); print(students["major"].value_counts())

housing BEFORE:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

housing_clean AFTER:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64

major BEFORE:
major
Cybersecurity             530
Data Science              285
Information Science       285
Information Technology    283
Business Analytics        255
Learning Technologies     210
Cyber Security             76
DataScience                39
Information  Science       32
Info Technology            32
Name: count, dtype: int64

major AFTER:
major
Cybersecurity             606
Data Science              324
Information Science       317
Information Technology    315
Business Analytics        255
Learning Technologies     210
Name: count, dtype: int64


In [6]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes

In [7]:
print(students[["age", "work_hours_per_week", "commute_miles", "credits_attempted", "final_gpa"]].describe().loc[["min", "max"]].T)

bad_age  = (students["age"] < 16) | (students["age"] > 100)               # a student can't be negative, 0-3, or 199 years old
bad_work = (students["work_hours_per_week"] < 0) | (students["work_hours_per_week"] > 168)   # >168 h = more than a week
impossible_ages = sorted(students.loc[bad_age, "age"].unique().tolist())
n_neg_work = int((students["work_hours_per_week"] < 0).sum())
print("impossible_ages:", impossible_ages, "| n_neg_work:", n_neg_work, "| rows with any impossible value:", (bad_age | bad_work).sum())

# Extreme-but-valid values that are KEPT
print("Kept extremes -> oldest valid ages:", sorted(students.loc[~bad_age, "age"])[-4:],
      "| max work hrs:", students["work_hours_per_week"].max(),
      "| max commute mi:", students["commute_miles"].max(),
      "| GPA == 0 rows:", (students["final_gpa"] == 0).sum())

                      min    max
age                 -22.0  220.0
work_hours_per_week -12.0   40.0
commute_miles         0.1   79.1
credits_attempted     0.0  128.0
final_gpa             0.0    4.0
impossible_ages: [-22, 0, 1, 3, 199, 220] | n_neg_work: 4 | rows with any impossible value: 10
Kept extremes -> oldest valid ages: [37, 38, 38, 39] | max work hrs: 40 | max commute mi: 79.1 | GPA == 0 rows: 43


In [8]:
# FIX, don't remove: the rest of each affected row is valid, and dropping students would break the roster.
# Impossible cell -> NaN -> column median (robust to the remaining extremes), with a flag column for traceability.
students["impossible_value_fixed"] = (bad_age | bad_work).astype(int)
students.loc[bad_age,  "age"] = np.nan
students.loc[bad_work, "work_hours_per_week"] = np.nan
students["age"] = students["age"].fillna(students["age"].median())
students["work_hours_per_week"] = students["work_hours_per_week"].fillna(students["work_hours_per_week"].median())
print(students[["age", "work_hours_per_week"]].describe().loc[["min", "max"]])

      age  work_hours_per_week
min  18.0                  0.0
max  39.0                 40.0


**Errors vs. extremes:** an *impossible* value (age −22, 0, 199, 220; work hours −12) violates a hard rule of the real world — no valid measurement could produce it, so it is a recording error and is fixed (set to the median, flagged). An *extreme but valid* value (a 79-mile commute, a 40-hour work week, a 39-year-old student, a GPA of 0.0) is unusual yet physically possible and carries real information, so it is kept.

In [9]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates

In [10]:
students["student_id"] = students["student_id"].str.strip().str.upper()
print("Exact duplicate rows:", students.duplicated().sum(), "| repeated student_id:", students["student_id"].duplicated().sum())
# 27 repeated IDs but only 18 exact copies: 9 are NEAR-duplicates (e.g. commute 2.1 vs 2.2), so de-dupe on the ID, not the whole row
students_dedup = students.drop_duplicates(subset="student_id", keep="first").reset_index(drop=True)
print("rows:", len(students), "->", len(students_dedup), "| unique IDs:", students_dedup["student_id"].nunique())

Exact duplicate rows: 18 | repeated student_id: 27
rows: 2027 -> 2000 | unique IDs: 2000


**Why not de-dupe enroll / activity? (1 sentence):** In `student_records` one row = one student, so a repeated ID is a copy; in `course_enrollments` one row = one student in one course in one term and in `weekly_activity` one row = one student in one week, so repeated IDs there are different real events (we confirmed there are zero duplicates at the true grain, below) and removing them would destroy data.

In [11]:
print("enroll duplicates at grain (sid, course, term):", enroll.duplicated(["sid", "course", "term"]).sum())
print("activity duplicates at grain (student_id, week):", activity.duplicated(["student_id", "week"]).sum())

enroll duplicates at grain (sid, course, term): 0
activity duplicates at grain (student_id, week): 0


In [12]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate

In [13]:
# The key: students/activity use "NU-101508"; enroll uses the bare integer 101508 -> standardize everything to an int `key`
students_dedup["key"] = students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)
activity["key"]       = activity["student_id"].str.strip().str.upper().str.replace("NU-", "", regex=False).astype(int)
enroll["key"]         = enroll["sid"].astype(int)

# Collapse each many-rows-per-student file to ONE row per student BEFORE joining (prevents row explosion)
activity_sum = activity.groupby("key").agg(total_minutes_active=("minutes_active", "sum"),
                                           total_lms_clicks=("lms_clicks", "sum"),
                                           weeks_recorded=("week", "nunique")).reset_index()
enroll["grade_points"] = enroll["grade"].map({"A": 4, "B": 3, "C": 2, "D": 1, "F": 0})   # W = withdrawn, no grade points
enroll_sum = enroll.groupby("key").agg(n_courses=("course", "size"),
                                       n_withdrawn=("grade", lambda g: int((g == "W").sum())),
                                       avg_grade_points=("grade_points", "mean")).reset_index()

# LEFT joins from the cleaned roster so no student can be lost or duplicated
analysis = (students_dedup
            .merge(enroll_sum,   on="key", how="left", indicator="_enr")
            .merge(activity_sum, on="key", how="left", indicator="_act"))
print(analysis.shape)

(2000, 24)


In [14]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join

In [15]:
keys = set(students_dedup["key"])
matched = int(enroll["sid"].isin(keys).sum())
print("enroll rows matching a roster student:", matched, "of", len(enroll), "| unmatched rows:", len(enroll) - matched)
print("Orphan IDs in enroll (not on roster):   ", len(set(enroll["key"]) - keys), "->", sorted(set(enroll["key"]) - keys)[:5], "...")
print("Orphan IDs in activity (not on roster): ", len(set(activity["key"]) - keys))
print("Roster students matched in enroll_sum:  ", (analysis["_enr"] == "both").sum(), "of", len(analysis))
print("Roster students matched in activity_sum:", (analysis["_act"] == "both").sum(), "of", len(analysis))
print("Final rows:", len(analysis), "== cleaned roster:", len(students_dedup), "->", len(analysis) == len(students_dedup))
assert len(analysis) == len(students_dedup) and analysis["key"].is_unique
analysis = analysis.drop(columns=["_enr", "_act"])

enroll rows matching a roster student: 8041 of 8088 | unmatched rows: 47
Orphan IDs in enroll (not on roster):    14 -> [100762, 101211, 101681, 102552, 102617] ...
Orphan IDs in activity (not on roster):  0
Roster students matched in enroll_sum:   2000 of 2000
Roster students matched in activity_sum: 2000 of 2000
Final rows: 2000 == cleaned roster: 2000 -> True


In [16]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates

In [17]:
raw_dates = analysis["enrollment_date"]
print("NaT/blank BEFORE:", raw_dates.isna().sum(), "| formats present:", raw_dates.str.replace(r"\d", "9", regex=True).str.replace(r"[A-Za-z]{3}", "Mon", regex=True).unique())
# Three formats are mixed (2023-09-13, 11/16/2022, 26-Mar-2024). No slash date has a first field > 12, so slashes are month-first.
dates_parsed = pd.to_datetime(raw_dates, format="mixed", errors="coerce")
print("NaT AFTER:", dates_parsed.isna().sum(), "-> no valid date lost" if dates_parsed.isna().sum() == raw_dates.isna().sum() else "-> LOST")
print("naive parse (default format) would lose:", pd.to_datetime(raw_dates, format="%Y-%m-%d", errors="coerce").isna().sum(), "dates")
print(dates_parsed.min().date(), "->", dates_parsed.max().date())
analysis["enrollment_date"] = dates_parsed

NaT/blank BEFORE: 0 | formats present: <StringArray>
['9999-99-99', '99/99/9999', '99-Mon-9999']
Length: 3, dtype: str
NaT AFTER: 0 -> no valid date lost
naive parse (default format) would lose: 1449 dates
2022-08-23 -> 2025-01-22


In [18]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize

In [19]:
# z-score of study_hours_reported; GPA bands 0-1 / 1-2 / 2-3 / 3-4
analysis["study_z"]  = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()
analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3, 4], labels=["0-1", "1-2", "2-3", "3-4"])
print(analysis["study_z"].agg(["mean", "std"]).round(3).to_dict())
print(analysis["gpa_band"].value_counts().sort_index())

{'mean': -0.0, 'std': 1.0}
gpa_band
0-1      70
1-2     528
2-3    1102
3-4     300
Name: count, dtype: int64


In [20]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file

In [21]:
analysis.to_csv("northgate_clean.csv", index=False)     # new file; raw CSVs are never written
print("wrote northgate_clean.csv", analysis.shape)
print("remaining nulls:", analysis.isna().sum()[lambda s: s > 0].to_dict())

wrote northgate_clean.csv (2000, 24)
remaining nulls: {'avg_grade_points': 1}


In [22]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log

**Cleaning log** (decision → justification)

- **Missing `study_hours_reported` (255 of 2,027, 12.6%) → regression imputation (Han's "most probable value") from `final_gpa` + `advisor_meetings`, with an `study_hours_imputed` flag.** Missing students have lower GPA (not random) and study hours correlate ~0.70 with GPA, so a global mean would flatten the relationship and dropping rows would bias the sample toward high-GPA students.
- **`housing` (8 spellings) → 3 groups (On-Campus / Off-Campus / With Family) via strip + lowercase + separator normalisation.** The variants were only case/hyphen/space differences of the same three real categories; `major` had the same defect and was fixed the same way.
- **Impossible ages (−22, 0, 1, 3, 199, 220) and negative work hours (−12 etc.) → set to the column median and flagged (`impossible_value_fixed`), not dropped.** They break hard real-world rules (recording errors), but the rest of each row is valid and the roster must stay intact.
- **Extreme-but-valid values (79-mile commute, 40 work hrs, age 39, GPA 0.0) → kept.** They are possible and informative; removing them would bias the analysis.
- **27 duplicate student rows → removed with `drop_duplicates(subset="student_id")`, keeping the first (2,027 → 2,000).** Only 18 are exact copies; 9 differ in a trivial field (e.g., commute 2.1 vs 2.2), so whole-row de-duplication would miss them.
- **`course_enrollments` and `weekly_activity` → not de-duplicated.** One row = one student-course-term / one student-week, so repeated IDs are legitimate events; there are 0 duplicates at the true grain.
- **Key: `"NU-101508"` vs bare int `sid` → all three files standardised to an integer `key`.** Formats differ only by the prefix, so the join would otherwise match nothing.
- **Join: many-row files aggregated to one row per student first, then LEFT-joined onto the cleaned roster.** Aggregating first prevents row explosion and a left join guarantees the table equals the roster (2,000 rows); 14 orphan enrollment IDs (47 rows) not on the roster are excluded by design.
- **`enrollment_date` (3 mixed formats) → parsed with `format="mixed"`; 0 blanks before and after.** Slash dates are month-first (no first field exceeds 12), and a single-format parse would have silently lost 1,449 dates.
- **`study_z` = z-score of study hours; `gpa_band` = 4 bands (0–1, 1–2, 2–3, 3–4).** The z-score puts study hours on a unit-free scale; bands match the framework's requested discretisation.


### D11 · Payoff

In [23]:
# Mean GPA and a relationship, raw vs clean
raw_gpa, clean_gpa = students_raw["final_gpa"].mean(), analysis["final_gpa"].mean()
print(f"Mean GPA   raw: {raw_gpa:.3f} (n={len(students_raw)})   clean: {clean_gpa:.3f} (n={len(analysis)})")

pairs = students_raw[["study_hours_reported", "final_gpa"]].dropna()
raw_r, clean_r = pairs.corr().iloc[0, 1], analysis["study_hours_reported"].corr(analysis["final_gpa"])
print(f"corr(study hours, GPA)  raw: {raw_r:.3f} (n={len(pairs)})   clean: {clean_r:.3f} (n={len(analysis)})")

print("\nDropout rate by GPA band (clean):"); print(analysis.groupby("gpa_band", observed=True)["dropped_out"].mean().round(3))
print("\nMean GPA by housing, raw (messy labels) vs clean:")
print(students_raw.groupby("housing")["final_gpa"].mean().round(3)); print(analysis.groupby("housing_clean")["final_gpa"].mean().round(3))
print("\ncorr(total_minutes_active, GPA) clean:", round(analysis["total_minutes_active"].corr(analysis["final_gpa"]), 3))

Mean GPA   raw: 2.307 (n=2027)   clean: 2.305 (n=2000)
corr(study hours, GPA)  raw: 0.696 (n=1772)   clean: 0.734 (n=2000)

Dropout rate by GPA band (clean):
gpa_band
0-1    0.243
1-2    0.295
2-3    0.196
3-4    0.110
Name: dropped_out, dtype: float64

Mean GPA by housing, raw (messy labels) vs clean:
housing
 On-Campus     2.157
Off-Campus     2.333
Off-campus     2.361
On-Campus      2.270
With Family    2.305
off campus     2.309
on-campus      2.400
with family    2.234
Name: final_gpa, dtype: float64
housing_clean
Off-Campus     2.330
On-Campus      2.276
With Family    2.293
Name: final_gpa, dtype: float64

corr(total_minutes_active, GPA) clean: 0.796


**Payoff.** On the clean table (n = 2,000) the mean GPA is **2.305**, and the most interesting relationship is that engagement tracks performance: `total_minutes_active` correlates **0.80** with `final_gpa`, study hours correlate **0.73**, and dropout falls from ~29% in the 1–2 GPA band to ~11% in the 3–4 band. **How cleaning changed the picture:** the mean GPA barely moved (2.307 → 2.305, removing 27 duplicates), but the raw `housing` column split into 8 spellings that hid the real three-group comparison (raw groups ranged 2.16–2.40 on tiny label-fragments vs. 2.28–2.33 for the true groups), and imputing the 255 blanks lifted the study-hours/GPA correlation from 0.70 (n = 1,772) to 0.73 (n = 2,000) — partly an artifact, since regression imputation by construction follows the GPA relationship, so that gain should be read cautiously.


In [24]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
